# XDF Processing and Epoch Creation for Hyperscanning Data

This notebook loads XDF format hyperscanning data (containing EEG recordings from two participants), extracts the data, creates epochs, and saves them in a pickle file for further processing.

Steps:
1. Find and load XDF files from the recording session
2. Identify the EEG streams for both participants
3. Create MNE Raw objects for each participant
4. Filter the data and create epochs
5. Save the epochs in a pickle file for hyperscanning analysis

In [ ]:
import pyxdf
import numpy as np
import mne
import os
import pickle
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
# Define the XDF directory and filename
xdf_directory = "./data/RecordedSession"
xdf_filename = "RetestXDFwMvmt-Laptops.xdf"  # Replace with your actual filename (e.g., "recording_session1.xdf")

# Create the full path by joining directory and filename
xdf_file = os.path.join(xdf_directory, xdf_filename)

# Check if the file exists
if os.path.exists(xdf_file):
    print(f"XDF file found: {xdf_file}")
else:
    print(f"Warning: XDF file not found: {xdf_file}")
    
    # Optionally, show available XDF files in the directory
    xdf_files = glob.glob(os.path.join(xdf_directory, "*.xdf"))
    if xdf_files:
        print(f"Available XDF files in the directory: {xdf_files}")

In [ ]:
# Load the XDF file using pyxdf
try:
    print(f"Loading XDF file: {xdf_file}...")
    streams, header = pyxdf.load_xdf(xdf_file)
        
    print("XDF file loaded successfully.")
    print(f"Found {len(streams)} streams")
    
    # Print information about each stream to help identify what we're working with
    for i, stream in enumerate(streams):
        print(f"Stream {i}:")
        print(f"  Name: {stream.get('info', {}).get('name', ['Unknown'])[0]}")
        print(f"  Type: {stream.get('info', {}).get('type', ['Unknown'])[0]}")
        print(f"  Channel count: {stream.get('info', {}).get('channel_count', ['Unknown'])[0]}")
        print(f"  Sampling rate: {stream.get('info', {}).get('nominal_srate', ['Unknown'])[0]}")
        print(f"  Data shape: {np.array(stream['time_series']).shape if 'time_series' in stream else 'Unknown'}")
except Exception as e:
    print(f"Error loading XDF file: {e}")
    import traceback
    traceback.print_exc()

## Identifying EEG Streams for Participants

Now that we've examined all streams in the XDF file, we need to identify which streams contain the EEG data for our two participants.

The streams with type 'EEG' typically contain the actual EEG recording data. For a hyperscanning experiment, we expect to have two EEG streams, one for each participant.

In [ ]:
# Identify the EEG streams for both participants
eeg_streams = []
for i, stream in enumerate(streams):
    if stream.get('info', {}).get('type', [''])[0] == 'EEG':
        eeg_streams.append((i, stream))

print(f"Found {len(eeg_streams)} EEG streams")

if len(eeg_streams) < 2:
    print(f"Warning: Only {len(eeg_streams)} EEG streams found. Two participants are expected.")

In [ ]:
# Process each EEG stream and create MNE Raw objects for each participant
raw_objects = []  # To store the Raw objects for both participants

for participant_id, (index, eeg_stream) in enumerate(eeg_streams, 1):
    print(f"\nProcessing data for participant {participant_id} (Stream {index}):")
    
    # Extract EEG data
    data = np.array(eeg_stream["time_series"])
    print(f"EEG data shape: {data.shape}")
    
    # Get sampling frequency
    sfreq = float(eeg_stream["info"]["nominal_srate"][0])
    if sfreq <= 0:
        print(f"Warning: Sampling rate is not positive. Using a default value of 250 Hz.")
        sfreq = 250.0
    
    # Use hardcoded channel names from mBrainTrain EEG16 device
    # These are the names we identified in the metadata you shared
    if data.shape[1] == 42:  # If we have the expected number of channels
        ch_names = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
                    'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'Fz', 'Cz', 'Pz', 'POz',
                    'FC1', 'FC2', 'CP1', 'CP2', 'FC5', 'FC6', 'CP5', 'CP6', 
                    'FT9', 'FT10', 'TP9', 'TP10',
                    'AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ',
                    'QuatW', 'QuatX', 'QuatY', 'QuatZ']
        
        # Assign appropriate channel types
        ch_types = ['eeg'] * 32 + ['misc'] * 10  # 32 EEG channels + 10 sensor channels
        
        print(f"Using known mBrainTrain EEG16 channel names")
    else:
        print(f"Unexpected number of channels: {data.shape[1]}")
        print("Using generic channel names instead")
        ch_names = [f'EEG{i+1}' for i in range(data.shape[1])]
        ch_types = ['eeg'] * len(ch_names)
    
    # Create MNE info object with the proper channel names
    info = mne.create_info(ch_names, sfreq, ch_types)
    
    # Create Raw object - transpose the data if necessary
    if data.shape[0] > data.shape[1]:  # If rows > columns (samples > channels)
        # Data format is samples x channels, needs to be channels x samples for MNE
        raw = mne.io.RawArray(data.T, info)
    else:
        # Data is already in channels x samples format
        raw = mne.io.RawArray(data, info)
    
    print(f"Created Raw object with {len(ch_names)} channels at {sfreq} Hz")
    
    # Apply standard 10-20 montage
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage, on_missing='ignore')
        print("Applied standard 10-20 montage")
    except Exception as e:
        print(f"Warning: Could not set montage: {e}")
    
    # Store the Raw object
    raw_objects.append(raw)

In [ ]:
# Optionally, apply standard montage and plot a segment of the data
for participant_id, raw in enumerate(raw_objects, 1):
    print(f"\nApplying montage and plotting data for participant {participant_id}:")
    
    # Apply a standard montage
    try:
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage, on_missing='ignore')
        print("Standard montage applied to XDF data.")
    except Exception as e:
        print(f"Warning: Could not set montage: {e}")
    
    # Plot a short segment of the raw data
    fig = raw.plot(n_channels=min(8, len(raw.ch_names)), duration=5, 
                 title=f"Participant {participant_id} - Raw EEG Data", 
                 scalings='auto', show=False)
    plt.tight_layout()
    plt.show()

In [ ]:
# Create epochs from the raw data for both participants using time-based approach
epochs_objects = []

for participant_id, raw in enumerate(raw_objects, 1):
    print(f"\nCreating epochs for participant {participant_id} using time-based approach:")
    
    # First, select only EEG channels to ensure consistent data format
    raw_eeg = raw.copy().pick_types(eeg=True, exclude=[])
    print(f"Selected {len(raw_eeg.ch_names)} EEG channels for analysis")
    
    # Skip filtering for now as requested
    raw_filtered = raw_eeg.copy.filter(l_freq=1.0, h_freq=40.0)
    #raw_filtered = raw_eeg.copy()
    
    # Create artificial events every 1 second
    event_duration = int(raw_eeg.info['sfreq'])  # 1 second in samples
    n_samples = len(raw_eeg.times)
    
    # Add a safety margin to avoid edge effects
    start_offset = int(2 * raw_eeg.info['sfreq'])  # 2 seconds from the beginning
    end_offset = int(2 * raw_eeg.info['sfreq'])    # 2 seconds from the end
    
    # Create events array - one event per second, avoiding edge effects
    usable_samples = n_samples - start_offset - end_offset
    n_events = usable_samples // event_duration
    events = np.zeros((n_events, 3), dtype=int)
    
    # Set event times and IDs
    for i in range(n_events):
        events[i, 0] = start_offset + (i * event_duration)  # Event sample with offset
        events[i, 2] = 1                                    # Event ID (using 1 for all events)
    
    print(f"Created {len(events)} artificial events spaced 1 second apart")
    
    # Create epochs from the filtered data
    tmin = -0.1  # Start 100ms before the event (short to avoid baseline issues)
    tmax = 0.9   # End 900ms after the event (total 1 second epochs)
    
    try:
        epochs = mne.Epochs(
            raw_filtered,
            events,
            tmin=tmin,
            tmax=tmax,
            baseline=(tmin, 0),  # Baseline correction using pre-stimulus period
            preload=True,
            reject=None,         # No automatic rejection
            flat=None,           # No flat channel detection
            reject_by_annotation=False
        )
        
        print(f"Created {len(epochs)} epochs for participant {participant_id}")
        epochs_objects.append(epochs)
        
        # Plot a few epochs to visualize
        if len(epochs) > 0:
            n_epochs_to_plot = min(5, len(epochs))
            n_channels_to_plot = min(8, len(epochs.ch_names))
            
            fig = epochs.plot(n_epochs=n_epochs_to_plot, 
                             n_channels=n_channels_to_plot, 
                             title=f"Participant {participant_id} - Epochs", 
                             scalings='auto', 
                             show=False)
            plt.tight_layout()
            plt.show()
        
    except Exception as e:
        print(f"Error creating epochs for participant {participant_id}: {e}")
        import traceback
        traceback.print_exc()
        
        # Create an empty epochs object as placeholder
        n_times = int(raw_eeg.info['sfreq'] * (tmax - tmin))
        empty_data = np.zeros((1, len(raw_eeg.ch_names), n_times))
        empty_info = raw_eeg.info.copy()
        epochs_objects.append(mne.EpochsArray(empty_data, empty_info))

In [ ]:
# Now save the epoch data to a pickle file as requested
try:
    # Create a dictionary with all the data we want to save
    if len(epochs_objects) >= 2:
        data_to_save = {
            'epo1_clean': epochs_objects[0],
            'epo2_clean': epochs_objects[1],
        }
        
        # Save the data to a pickle file
        output_filename = './data/generated/hyperscanning_recorded_data.pkl'
        with open(output_filename, 'wb') as f:
            pickle.dump(data_to_save, f)
        print(f"Data saved successfully to '{output_filename}'")
    else:
        print(f"Warning: Not enough epoch objects to save (found {len(epochs_objects)}, need at least 2)")
        
except Exception as e:
    print(f"Error saving data to pickle file: {e}")

## Verification and Next Steps

The EEG data from both participants has been:
1. Extracted from the XDF file
2. Converted to MNE Raw objects 
3. Filtered and segmented into epochs
4. Saved in a pickle file (`hyperscanning_recorded_data.pkl`)

This pickle file contains:
- `epo1_clean`: Epochs for participant 1
- `epo2_clean`: Epochs for participant 2

The saved data can be loaded back with:

```python
import pickle
with open('./data/hyperscanning_recorded_data.pkl', 'rb') as f:
    saved_data = pickle.load(f)
    
# Extract the cleaned epochs from the loaded data
epo1_clean = saved_data['epo1_clean']
epo2_clean = saved_data['epo2_clean']